### Notes from Readings - Factors Contributing to Disturbance




In [ ]:
import pandas as pd
import geopandas as gpd
import plotly.express as px
import h3

from human.activity_and_effort.ais.analysis import (
    general_parquet_data_loader, 
    general_geoparquet_data_loader, 
    h3_to_polygon, 
    subset_polygon_for_check
)

In [ ]:
# ------------------------------------ #
#                Modules               # 

# Base Path 
base_path = '../..'

# AIS Path
ais_path = f'{base_path}/data/processed/ais/'

# Sightings Path
sightings_path = f'{base_path}/data/processed/domain/whale_layer/sightings/imputed_retrospective.parquet'
sightings_associations_path = f'{base_path}/data/processed/domain/whale_layer/sightings/associations.parquet'

# Water Path
water_path = f'{base_path}/data/processed/gis/marine/TERRITORIAL_WATER_POLYGON.parquet'

#                                      #
# ------------------------------------ #

In [ ]:
# Load Data
## AIS Data
ais_data = general_parquet_data_loader(ais_path)
if ais_data is not None:
    ais_data['DATE'] = pd.to_datetime(ais_data['DATE'])

## SIGHTINGS Data
sightings_data = general_parquet_data_loader(sightings_path)
if sightings_data is not None:
    sightings_data['TYPE'] = sightings_data['ECOTYPE_DETAIL_EFFECTIVE']
    sightings_data['DATETIME'] = pd.to_datetime(sightings_data['SIGHTING_DATE_UTC'])
    sightings_data['DATE'] = sightings_data['DATETIME']
    pod_associations = general_parquet_data_loader(sightings_associations_path)
    if pod_associations is not None:
        pod_associations = pod_associations[pod_associations['ASSOCIATION_KIND'].eq('POD')]
        pod_associations = pod_associations.groupby('OBSERVATION_ID', as_index=False)['ASSOCIATION_VALUE'].agg(','.join)
        pod_associations = pod_associations.rename(columns={'ASSOCIATION_VALUE': 'POD'})
        sightings_data = sightings_data.merge(pod_associations, on='OBSERVATION_ID', how='left')

## Territorial Waters AOI
water_geometry_data = general_geoparquet_data_loader(water_path)

## Subset Polygon
subset_area = subset_polygon_for_check()

In [ ]:
water_geometry_data = water_geometry_data.clip(subset_area)

In [ ]:
# Clip AIS Data to Subset Area
ais_data_h3 = ais_data[['H3_CELL']].drop_duplicates()
ais_data_h3['geometry'] = ais_data_h3['H3_CELL'].apply(lambda x: h3_to_polygon(x))
ais_data_h3 = gpd.GeoDataFrame(ais_data_h3, geometry = 'geometry', crs = 'EPSG:4326')

ais_data_h3 = ais_data_h3.clip(subset_area)

water_polygon = water_geometry_data.dissolve()
ais_data_h3 = ais_data_h3.clip(water_polygon)

h3_list = ais_data_h3['H3_CELL'].tolist()

In [ ]:
# Clip AIS Data to H3 Range in Subset Area
ais_data = ais_data[ais_data.H3_CELL.isin(h3_list)]

# Clip Sightings to AIS Time Span
ais_data_min_date = ais_data['DATE'].min()
ais_data_max_date = ais_data['DATE'].max()

sightings_data = sightings_data[
    (sightings_data.DATE >= ais_data_min_date) &
    (sightings_data.DATE <= ais_data_max_date)
    ]

sightings_data['H3_CELL'] = sightings_data.apply(lambda s: h3.latlng_to_cell(s["LATITUDE"], s["LONGITUDE"], 6), axis = 1)

# Clip Sightings to H# Grids Occupied by AIS
ais_data_h3_occupied = ais_data['H3_CELL'].unique().tolist()

sightings_data = sightings_data[sightings_data.H3_CELL.isin(ais_data_h3_occupied)]

In [ ]:
sightings_tt = sightings_data.groupby(['DATE', 'TYPE'], as_index = False).agg(COUNT = ('OBSERVATION_ID', 'nunique'))

In [ ]:
fig = px.line(sightings_tt, x = 'DATE', y = 'COUNT')
fig.show()

In [ ]:
# Get Count of Active - Map
df_counts = ais_data.groupby(['H3_CELL'], as_index = False)['MMSI'].count()
df_counts['geometry'] = df_counts['H3_CELL'].apply(lambda x: h3_to_polygon(x))
df_counts = gpd.GeoDataFrame(df_counts, geometry = 'geometry', crs = 'EPSG:4326')

df_counts.explore(
    column="MMSI",
    cmap="turbo",
    scheme="Quantiles",   # or "EqualInterval"
    k=5000
).save("./maps/ais/mmsi_count.html")

In [ ]:
# Get Vessel Count Over Time
vessel_count = ais_data.groupby('DATE', as_index = False)['MMSI'].nunique()

In [ ]:
fig = px.line(vessel_count, x = 'DATE', y = 'MMSI', title = 'Number of Unique Vessel Observations')
fig.show()

In [ ]:
# Get Active Vessel Grids Per Date
vessel_count_h3 = ais_data.groupby('DATE', as_index = False)['H3_CELL'].nunique()

In [ ]:
fig = px.line(vessel_count_h3, x = 'DATE', y = 'H3_CELL', title = 'Number of Active Grids Per Day')
fig.show()

In [ ]:
# Subset H3 Data to Kingston-Edmonds Area
h3_select = [
    '8628d57b7ffffff',
    '8628d57a7ffffff',
    '8628d545fffffff',
    '8628d579fffffff',
    '8628d5787ffffff'
 ]

In [ ]:
tmp_sights = sightings_data[sightings_data.H3_CELL.isin(h3_select)]

In [ ]:
tmp_ais = ais_data[ais_data['H3_CELL'].isin(h3_select)]
tmp_ais = tmp_ais[~tmp_ais.VesselType.isin([37])]

tmp_ais_count = tmp_ais.groupby(['DATE', 'VesselType'], as_index = False).agg(COUNT = ('MMSI', 'count'))
tmp_sights = tmp_sights[['DATE', 'POD']].drop_duplicates()

In [ ]:
# fig = px.line(tmp_ais_count[tmp_ais_count.VesselType.astype(str).str.startswith('8')], x = 'DATE', y = 'COUNT', color = 'VesselType')


# j_sights = tmp_sights[tmp_sights.POD == 'J']
# for i, row in j_sights.iterrows():
#     fig.add_vline(x = row['DATE'], line_color = '#e76f51')

# k_sights = tmp_sights[tmp_sights.POD == 'K']
# for i, row in k_sights.iterrows():
#     fig.add_vline(x = row['DATE'], line_color = '#e9c46a')

# l_sights = tmp_sights[tmp_sights.POD == 'L']
# for i, row in l_sights.iterrows():
#     fig.add_vline(x = row['DATE'], line_color = '#2a9d8f')

# # fig.update_layout(
# #     template="simple_white",
# #     xaxis_title="Date",
# #     yaxis_title=f"Count MMSI",
# #     title=f"AIS Signal Vs. Presence"
# # )

# fig.show()

In [ ]:
ais_data.VesselType.unique()

In [ ]:
# Let's get activity per hour of each dat within each h3 grid then calculate an average vesel pressure per cell per day (and show this over time...
ais_count_per_grid_hour = ais_data.groupby(['HOUR_BIN', 'H3_CELL'], as_index = False).agg(COUNT = ('MMSI', 'nunique'))


In [ ]:
ais_count_per_grid_hour['DATETIME'] = pd.to_datetime(ais_count_per_grid_hour['HOUR_BIN'])

In [ ]:
ais_count_per_grid_hour['DATE'] = ais_count_per_grid_hour['DATETIME'].dt.date
ais_count_per_grid_hour['HOUR'] = ais_count_per_grid_hour['DATETIME'].dt.hour

In [ ]:
unique_h3 = ais_count_per_grid_hour['H3_CELL'].unique().tolist()
ais_dates = ais_count_per_grid_hour[['DATE']].drop_duplicates()

ais_dates['H3_CELL'] = ais_dates.apply(lambda x: unique_h3, axis = 1)
ais_dates = ais_dates.explode('H3_CELL')
ais_dates['HOUR'] = ais_dates.apply(lambda x: list(range(1, 25)), axis = 1)
ais_dates = ais_dates.explode('HOUR')

ais_count_per_grid_hour = ais_count_per_grid_hour[['H3_CELL', 'DATE', 'HOUR', 'COUNT']]
ais_count_per_grid_hour = pd.merge(ais_count_per_grid_hour, ais_dates, how = 'outer', on = ['DATE', 'H3_CELL', 'HOUR'])
ais_count_per_grid_hour['COUNT'] = ais_count_per_grid_hour['COUNT'].fillna(0)

In [ ]:
# Get Mean Pressure Per Date
ais_mean_count_per_grid_hour = ais_count_per_grid_hour.groupby(['DATE', 'H3_CELL'], as_index = False).agg(MEAN_COUNT = ('COUNT', 'mean'))

In [ ]:
sightings_count = sightings_data.groupby(['H3_CELL', 'DATE'], as_index = False).agg(SIGHTINGS_COUNT = ('OBSERVATION_ID', 'nunique'))
sightings_count['DATE'] = sightings_count['DATE'].dt.date

In [ ]:
ais_mean_count_per_grid_hour = pd.merge(ais_mean_count_per_grid_hour, sightings_count, on = ['H3_CELL', 'DATE'], how = 'outer')
ais_mean_count_per_grid_hour['SIGHTINGS_COUNT'] = ais_mean_count_per_grid_hour['SIGHTINGS_COUNT'].fillna(0)

In [ ]:
px.scatter(ais_mean_count_per_grid_hour[ais_mean_count_per_grid_hour.SIGHTINGS_COUNT !=0], x = 'MEAN_COUNT', y = 'SIGHTINGS_COUNT')